# Notebook 02 — Embeddings et base vectorielle (ChromaDB)

**Module 4 · Capstone, étape 2.** Rapports annuels de RBC, TD, BMO, Scotiabank et CIBC (exercices 2024 et 2025).

**Entrée :** `data/processed/chunks.jsonl` (notebook 01).
**Sorties :** embeddings dans `models/`, collection ChromaDB, journal de recherche et precision@3 dans `reports/`.

| Consigne du sujet | Section |
|---|---|
| 1. Choisir un modèle d'embedding | 1 |
| 2. Embedder les chunks, noter temps et coût | 2 |
| 3-4. Collection ChromaDB avec métadonnées, insertion | 3 |
| 5. Dix requêtes de test, top-5 et MMR | 4 et 5 |
| 6. Precision@3 (vérification manuelle) | 6 et 7 |
| 7. Filtrage par métadonnées | 8 |
| 8. Comparaison avec OpenAI (optionnelle) | 9 |

**Pour relancer :** Restart puis Run All. Les embeddings sont rechargés depuis `models/` s'ils existent déjà (sinon le calcul prend environ 7 minutes).

## 0. Configuration

In [24]:
import json, time, re, os, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
REPORTS = ROOT / "reports"
for p in (MODELS, REPORTS):
    p.mkdir(exist_ok=True)

# Paramètres de l'expérience
MODEL_NAME = "all-MiniLM-L6-v2"
MAX_SEQ = 512                    # maximum de MiniLM (le défaut, 256, ne verrait qu'un tiers des chunks)
K, FETCH_K, LAMBDA = 5, 20, 0.7  # résultats retournés, candidats pour MMR, équilibre pertinence/diversité
SIM_MIN = 0.3                    # seuil de similarité : en dessous, on répond « aucune information »
COLLECTION = "bank_annual_reports_minilm"
FORCE_RECOMPUTE = False          # True pour recalculer les embeddings même si le cache est valide

## 1. Données et choix du modèle (consigne 1)

**Modèle retenu : `all-MiniLM-L6-v2`** — un modèle d'embedding (pas un LLM) de 384 dimensions, gratuit, exécuté localement. Le texte des rapports ne quitte pas la machine, ce qui compte pour la confidentialité (OSFI B-10). Limite : il lit au maximum 512 tokens, or nos chunks en font environ 800. On mesure l'effet ci-dessous.

In [25]:
chunks = pd.read_json(PROCESSED / "chunks.jsonl", lines=True)
required = ["chunk_id", "bank", "bank_code", "document_type", "fiscal_year", "quarter",
            "part", "section", "risk_category", "page_start", "page_end", "has_table", "n_tokens", "citation", "text"]
missing = [c for c in required if c not in chunks.columns]
assert not missing, f"Colonnes manquantes : {missing}. Relancer le notebook 01."
assert chunks.chunk_id.is_unique
assert chunks[required].isna().sum().sum() == 0, "Valeurs manquantes : ChromaDB refuse les métadonnées vides"

print(f"{len(chunks)} chunks chargés")
print(chunks.groupby(["bank_code", "fiscal_year"]).size().unstack())

3562 chunks chargés
fiscal_year  2024  2025
bank_code              
BMO           339   314
BNS           359   358
CIBC          354   332
RBC           365   328
TD            412   401


In [26]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME, device="cpu")
print("Limite par défaut :", model.max_seq_length, "tokens | limite physique :", model[0].auto_model.config.max_position_embeddings)
model.max_seq_length = MAX_SEQ

ids = model.tokenizer(chunks.text.tolist(), add_special_tokens=True, truncation=False)["input_ids"]
chunks["n_tokens_minilm"] = [len(x) for x in ids]

share_trunc_256 = (chunks.n_tokens_minilm > 256).mean()
share_trunc_512 = (chunks.n_tokens_minilm > MAX_SEQ).mean()
share_text_seen = np.minimum(chunks.n_tokens_minilm, MAX_SEQ).sum() / chunks.n_tokens_minilm.sum()

print(f"Taille moyenne d'un chunk avec le tokenizer de MiniLM : {chunks.n_tokens_minilm.mean():.0f} tokens")
print(f"Chunks tronqués à 256 : {share_trunc_256:.0%} | à 512 : {share_trunc_512:.0%}")
print(f"Part du texte réellement vue par MiniLM avec la limite de {MAX_SEQ} : {share_text_seen:.0%}")

Limite par défaut : 256 tokens | limite physique : 512


Token indices sequence length is longer than the specified maximum sequence length for this model (864 > 256). Running this sequence through the model will result in indexing errors


Taille moyenne d'un chunk avec le tokenizer de MiniLM : 724 tokens
Chunks tronqués à 256 : 99% | à 512 : 90%
Part du texte réellement vue par MiniLM avec la limite de 512 : 69%


## 2. Embeddings (consigne 2)

Les vecteurs sont **normalisés** (norme 1), donc cosinus = produit scalaire. Ils sont sauvegardés dans `models/` avec les identifiants des chunks et une empreinte du texte : si rien n'a changé, on les recharge au lieu de les recalculer.

In [27]:
text_hash = hashlib.md5("\n".join(chunks.text).encode("utf-8")).hexdigest()
emb_path = MODELS / "minilm_embeddings.npy"
ids_path = MODELS / "minilm_chunk_ids.csv"
cfg_path = MODELS / "minilm_config.json"

def cache_is_valid():
    if not (emb_path.exists() and ids_path.exists() and cfg_path.exists()):
        return False
    cfg = json.loads(cfg_path.read_text())
    same_ids = pd.read_csv(ids_path).chunk_id.tolist() == chunks.chunk_id.tolist()
    same_text = cfg.get("text_hash", text_hash) == text_hash     # anciens fichiers sans empreinte : acceptés
    return same_ids and same_text and cfg.get("model") == MODEL_NAME and cfg.get("max_seq_length") == MAX_SEQ

if cache_is_valid() and not FORCE_RECOMPUTE:
    emb = np.load(emb_path)
    elapsed = json.loads(cfg_path.read_text()).get("seconds", float("nan"))
    print(f"Embeddings rechargés depuis models/ : {emb.shape[0]} x {emb.shape[1]} (calculés à l'origine en {elapsed/60:.1f} min)")
else:
    t0 = time.time()
    emb = model.encode(chunks.text.tolist(), batch_size=32, normalize_embeddings=True,
                       show_progress_bar=True, convert_to_numpy=True)
    elapsed = time.time() - t0
    np.save(emb_path, emb)
    chunks[["chunk_id"]].to_csv(ids_path, index=False)
    cfg_path.write_text(json.dumps({
        "model": MODEL_NAME, "dim": int(emb.shape[1]), "max_seq_length": MAX_SEQ, "normalized": True,
        "n_chunks": int(emb.shape[0]), "seconds": round(elapsed, 1), "text_hash": text_hash,
        "share_chunks_truncated_512": round(float(share_trunc_512), 3), "share_text_seen_512": round(float(share_text_seen), 3),
    }, indent=2))
    print(f"{emb.shape[0]} embeddings de dimension {emb.shape[1]} en {elapsed/60:.1f} min ({emb.shape[0]/elapsed:.1f} chunks/s) | coût : 0 $ (local)")

norms = np.linalg.norm(emb, axis=1)
assert emb.shape[0] == len(chunks) and not np.isnan(emb).any()
print(f"Normes : min {norms.min():.4f}, max {norms.max():.4f}")

Embeddings rechargés depuis models/ : 3562 x 384 (calculés à l'origine en 6.8 min)
Normes : min 1.0000, max 1.0000


## 3. Base vectorielle ChromaDB (consignes 3 et 4)

Collection persistante (`models/chroma/`), similarité cosinus. On fournit nos propres vecteurs, pour que ChromaDB n'en recalcule pas avec un autre modèle. Métadonnées : banque, type de document, année fiscale, trimestre, partie, section, catégorie de risque, pages, citation.

In [28]:
import chromadb
from chromadb.config import Settings

def open_client():
    return chromadb.PersistentClient(path=str(MODELS / "chroma"), settings=Settings(anonymized_telemetry=False))

client = open_client()
if COLLECTION in [c.name for c in client.list_collections()]:      # on repart d'une collection vide à chaque exécution
    client.delete_collection(COLLECTION)
col = client.create_collection(COLLECTION, metadata={"hnsw:space": "cosine"})

meta_cols = ["bank", "bank_code", "document_type", "fiscal_year", "quarter", "part", "section",
             "risk_category", "page_start", "page_end", "has_table", "n_tokens", "citation"]
INT_COLS = ("fiscal_year", "page_start", "page_end", "n_tokens")
metadatas = [{k: (bool(v) if k == "has_table" else int(v) if k in INT_COLS else str(v)) for k, v in r.items()}
             for r in chunks[meta_cols].to_dict("records")]

t0 = time.time()
for i in range(0, len(chunks), 500):
    sl = slice(i, i + 500)
    col.add(ids=chunks.chunk_id.tolist()[sl], embeddings=emb[sl].tolist(),
            documents=chunks.text.tolist()[sl], metadatas=metadatas[sl])
print(f"{col.count()} chunks insérés en {time.time() - t0:.0f} s")

# Contrôles
assert col.count() == len(chunks)
assert open_client().get_collection(COLLECTION).count() == len(chunks), "La collection ne se rouvre pas depuis le disque"
probe = 1234                                    # un chunk doit être son propre plus proche voisin
res = col.query(query_embeddings=[emb[probe].tolist()], n_results=1, include=["distances"])
assert res["ids"][0][0] == chunks.chunk_id.iloc[probe] and res["distances"][0][0] < 1e-3
counts = {b: {y: len(col.get(where={"$and": [{"bank_code": b}, {"fiscal_year": y}]}, include=[])["ids"]) for y in (2024, 2025)}
          for b in ["RBC", "TD", "BMO", "BNS", "CIBC"]}
print("Chunks par banque et par année dans la collection :", counts)

3562 chunks insérés en 11 s
Chunks par banque et par année dans la collection : {'RBC': {2024: 365, 2025: 328}, 'TD': {2024: 412, 2025: 401}, 'BMO': {2024: 339, 2025: 314}, 'BNS': {2024: 359, 2025: 358}, 'CIBC': {2024: 354, 2025: 332}}


## 4. Fonctions de recherche

Règles appliquées (bonnes pratiques du cours) : **même modèle** pour indexer et pour interroger ; **k = 5** ; **top-k** pour les questions précises et **MMR (λ = 0,7)** pour les questions larges, les deux étant calculés pour chaque requête ; **seuil de similarité à 0,3** ; **chaque recherche est journalisée** (`reports/retrieval_log.jsonl`).

In [29]:
RETRIEVAL_LOG = []

def build_where(bank=None, year=None):
    conds = []
    if bank: conds.append({"bank_code": bank})
    if year: conds.append({"fiscal_year": year})
    if not conds: return None
    return conds[0] if len(conds) == 1 else {"$and": conds}

def embed_query(q):
    return model.encode([q], normalize_embeddings=True, convert_to_numpy=True)[0]

def to_rows(res):
    return [{"chunk_id": cid, "sim": 1 - d, "citation": m["citation"], "bank_code": m["bank_code"],
             "fiscal_year": m["fiscal_year"], "risk_category": m["risk_category"], "text": doc}
            for cid, doc, m, d in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0])]

def retrieve_topk(q, k=K, where=None):
    res = col.query(query_embeddings=[embed_query(q).tolist()], n_results=k, where=where,
                    include=["documents", "metadatas", "distances"])
    return to_rows(res)

def retrieve_mmr(q, k=K, fetch_k=FETCH_K, lam=LAMBDA, where=None):
    qv = embed_query(q)
    res = col.query(query_embeddings=[qv.tolist()], n_results=fetch_k, where=where,
                    include=["documents", "metadatas", "distances", "embeddings"])
    rows = to_rows(res)
    cand = np.array(res["embeddings"][0])
    rel = cand @ qv
    selected, remaining = [], list(range(len(rows)))
    while remaining and len(selected) < k:
        if not selected:
            best = max(remaining, key=lambda i: rel[i])
        else:
            sel = cand[selected]
            best = max(remaining, key=lambda i: lam * rel[i] - (1 - lam) * float(np.max(sel @ cand[i])))
        selected.append(best)
        remaining.remove(best)
    return [rows[i] for i in selected]

def search(qid, q, method, where=None):
    out = (retrieve_topk if method == "topk" else retrieve_mmr)(q, where=where)
    for rank, r in enumerate(out, 1):
        RETRIEVAL_LOG.append({"qid": qid, "method": method, "rank": rank, "chunk_id": r["chunk_id"],
                              "sim": round(r["sim"], 4), "kept": bool(r["sim"] >= SIM_MIN), "where": str(where), "query": q})
    kept = [r for r in out if r["sim"] >= SIM_MIN]
    if not kept:
        print(f"  [{qid}/{method}] No relevant information found (aucun chunk au-dessus de {SIM_MIN})")
    return kept

# Test du seuil de similarité
for q in ["What is Desjardins' CET1 ratio?", "What is the capital of France?"]:
    r = retrieve_topk(q, k=3)
    print(f"{q!r} -> similarité max {r[0]['sim']:.3f} | au-dessus du seuil : {sum(x['sim'] >= SIM_MIN for x in r)}/3 | meilleur : {r[0]['citation'][:55]}")

"What is Desjardins' CET1 ratio?" -> similarité max 0.408 | au-dessus du seuil : 3/3 | meilleur : Bank of Montreal – Annual Report FY2024 – MD&A (p.20-21
'What is the capital of France?' -> similarité max 0.185 | au-dessus du seuil : 0/3 | meilleur : Bank of Montreal – Annual Report FY2024 – MD&A (p.26-27


## 5. Dix requêtes de test (consigne 5)

Dix questions sur les 5 banques, les 2 années et plusieurs risques, écrites **avant** de voir les résultats. La Q6 est volontairement sans filtre (toutes les banques).
- **v1 :** la question telle qu'un analyste l'écrirait, avec le nom de la banque.
- **v2 :** la même question **sans le nom de la banque** (le filtre de métadonnées garantit déjà la banque et l'année). La v2 a été écrite après avoir vu les échecs de la v1 (voir section 7).

In [30]:
QUERIES = [
    dict(id=1,  q="What was RBC's provision for credit losses in fiscal 2025?",                                   type="factual", bank="RBC",  year=2025, kw=r"provision for credit losses|\bPCL\b"),
    dict(id=2,  q="What was TD's Liquidity Coverage Ratio as at October 31, 2025?",                                type="factual", bank="TD",   year=2025, kw=r"liquidity coverage ratio|\bLCR\b"),
    dict(id=3,  q="How does BMO describe its approach to climate risk management?",                               type="broad",   bank="BMO",  year=2025, kw=r"climate"),
    dict(id=4,  q="What does Scotiabank say about cybersecurity and technology risk?",                             type="broad",   bank="BNS",  year=2025, kw=r"cyber|technology risk"),
    dict(id=5,  q="What was CIBC's CET1 capital ratio at October 31, 2024?",                                       type="factual", bank="CIBC", year=2024, kw=r"\bCET1\b"),
    dict(id=6,  q="Which banks discuss artificial intelligence or OSFI Guideline E-23 on model risk?",             type="broad",   bank=None,   year=None, kw=r"artificial intelligence|E-23|model risk"),
    dict(id=7,  q="What were RBC's gross impaired loans in fiscal 2024?",                                          type="factual", bank="RBC",  year=2024, kw=r"gross impaired loans|impaired loans"),
    dict(id=8,  q="How sensitive are TD's net interest income and economic value of equity to interest rate changes?", type="factual", bank="TD", year=2024, kw=r"economic value of equity|\bEVE\b"),
    dict(id=9,  q="How is the credit quality of BMO's residential mortgage portfolio?",                            type="broad",   bank="BMO",  year=2024, kw=r"mortgage"),
    dict(id=10, q="What emerging risks does Scotiabank identify for 2024?",                                        type="broad",   bank="BNS",  year=2024, kw=r"emerging risk"),
]
QUERIES_V2 = {   # même question sans le nom de la banque (Q6 inchangée : pas de banque dans la question)
    1:  "What was the provision for credit losses in fiscal 2025?",
    2:  "What was the Liquidity Coverage Ratio as at October 31, 2025?",
    3:  "How does the bank describe its approach to climate risk management?",
    4:  "What does the bank say about cybersecurity and technology risk?",
    5:  "What was the CET1 capital ratio at October 31, 2024?",
    7:  "What were the gross impaired loans in fiscal 2024?",
    8:  "How sensitive are net interest income and economic value of equity to interest rate changes?",
    9:  "How is the credit quality of the residential mortgage portfolio?",
    10: "What emerging risks does the bank identify for 2024?",
}

results, results_v2, rows = {}, {}, []
for Q in QUERIES:
    where = build_where(Q["bank"], Q["year"])
    for method in ("topk", "mmr"):
        results[(Q["id"], method)] = search(Q["id"], Q["q"], method, where)
        results_v2[(Q["id"], method)] = search(f"{Q['id']}-v2", QUERIES_V2.get(Q["id"], Q["q"]), method, where)
        res = results[(Q["id"], method)]
        rows.append({"Q": Q["id"], "type": Q["type"], "filtre": f"{Q['bank'] or 'toutes'} {Q['year'] or ''}".strip(), "méthode": method,
                     "résultats": len(res), "sim max": round(max((r["sim"] for r in res), default=0), 3),
                     "sim moy": round(float(np.mean([r["sim"] for r in res])) if res else 0, 3),
                     "banques": len({r["bank_code"] for r in res})})
print("Version v1 (avec le nom de la banque) :")
print(pd.DataFrame(rows).to_string(index=False))

print("\nChunks en commun entre top-k et MMR (sur 5), v1 :",
      {f"Q{Q['id']}": len({r['chunk_id'] for r in results[(Q['id'], 'topk')]} & {r['chunk_id'] for r in results[(Q['id'], 'mmr')]}) for Q in QUERIES})
print("Chunks identiques entre v1 et v2, top-3 top-k :",
      {f"Q{Q['id']}": len({r['chunk_id'] for r in results[(Q['id'], 'topk')][:3]} & {r['chunk_id'] for r in results_v2[(Q['id'], 'topk')][:3]}) for Q in QUERIES})

Version v1 (avec le nom de la banque) :
 Q    type    filtre méthode  résultats  sim max  sim moy  banques
 1 factual  RBC 2025    topk          5    0.520    0.476        1
 1 factual  RBC 2025     mmr          5    0.520    0.475        1
 2 factual   TD 2025    topk          5    0.450    0.413        1
 2 factual   TD 2025     mmr          5    0.450    0.410        1
 3   broad  BMO 2025    topk          5    0.577    0.559        1
 3   broad  BMO 2025     mmr          5    0.577    0.553        1
 4   broad  BNS 2025    topk          5    0.583    0.541        1
 4   broad  BNS 2025     mmr          5    0.583    0.536        1
 5 factual CIBC 2024    topk          5    0.512    0.503        1
 5 factual CIBC 2024     mmr          5    0.512    0.474        1
 6   broad    toutes    topk          5    0.604    0.573        2
 6   broad    toutes     mmr          5    0.604    0.557        3
 7 factual  RBC 2024    topk          5    0.580    0.527        1
 7 factual  RBC 2024  

## 6. Jugement manuel et precision@3 (consigne 6)

**Règle de jugement (appliquée partout) :** un chunk est **pertinent** s'il contient l'information qui répond à la question — le chiffre demandé pour une question à chiffre, une description substantielle du sujet pour une question descriptive. Une simple mention en passant, ou la bonne banque et la bonne année sans la réponse, comptent comme **non pertinent**.

Les verdicts sont rangés **par question**, car un même chunk peut répondre à une question et pas à une autre. Ils ont été posés à partir de passages lus (extraits et recherches de mots-clés dans les chunks) et **doivent être relus** ; les cas limites sont signalés en commentaire. Pour juger un nouveau chunk : l'ajouter dans `JUDGE`, relancer la cellule suivante.

In [31]:
def snippet(text, kw, width=280):
    m = re.search(kw, text, re.I)
    s = max(0, m.start() - 100) if m else 0
    return " ".join(text[s:s + width].split())

def show(chunk_id):
    print(chunks.loc[chunks.chunk_id == chunk_id].iloc[0].text)

def grep_chunk(cid, pattern, n=4, before=120, after=260):
    t = chunks.loc[chunks.chunk_id == cid].iloc[0].text
    ms = list(re.finditer(pattern, t, re.I))
    print(f"\n######## {cid} | {len(ms)} occurrence(s) de /{pattern}/")
    for m in ms[:n]:
        s = max(0, m.start() - before)
        print("   …" + " ".join(t[s:m.end() + after].split()) + "…")

In [32]:
# True = le chunk contient la réponse, False = non. Format : JUDGE[numéro de question][identifiant du chunk]
JUDGE = {
    1: {"RBC_annual_report_FY2025-0235": False, "RBC_annual_report_FY2025-0141": False,
        "RBC_annual_report_FY2025-0160": False, "RBC_annual_report_FY2025-0246": False,
        "RBC_annual_report_FY2025-0237": False, "RBC_annual_report_FY2025-0305": False,
        "RBC_annual_report_FY2025-0188": False,
        "RBC_annual_report_FY2025-0151": False},      # cas limite : "Total PCL on impaired loans 3,734" et "... performing loans 622" y sont, pas leur somme
    2: {"TD_annual_report_FY2025-0028": False, "TD_annual_report_FY2025-0396": False,
        "TD_annual_report_FY2025-0114": False, "TD_annual_report_FY2025-0086": False,
        "TD_annual_report_FY2025-0181": True,         # tableau : « Liquidity coverage ratio 130% 138% »
        "TD_annual_report_FY2025-0182": True},        # « The Bank's average LCR was 130% for the quarter ended October 31, 2025 »
    3: {"BMO_annual_report_FY2025-0148": True, "BMO_annual_report_FY2025-0150": True,
        "BMO_annual_report_FY2025-0151": True, "BMO_annual_report_FY2025-0084": True,
        "BMO_annual_report_FY2025-0146": False, "BMO_annual_report_FY2025-0147": False},
    4: {"BNS_annual_report_FY2025-0357": False, "BNS_annual_report_FY2025-0007": False,
        "BNS_annual_report_FY2025-0103": False, "BNS_annual_report_FY2025-0160": False,
        "BNS_annual_report_FY2025-0108": True, "BNS_annual_report_FY2025-0112": True,
        "BNS_annual_report_FY2025-0111": False,
        "BNS_annual_report_FY2025-0006": False},      # une seule mention en passant (comité du conseil)
    5: {"CIBC_annual_report_FY2024-0350": False, "CIBC_annual_report_FY2024-0288": False,
        "CIBC_annual_report_FY2024-0073": False, "CIBC_annual_report_FY2024-0256": False,
        "CIBC_annual_report_FY2024-0075": False},     # donne la variation (+0,9 %) mais pas le niveau du ratio
    6: {"TD_annual_report_FY2025-0173": True, "TD_annual_report_FY2025-0016": False,
        "BNS_annual_report_FY2025-0112": True,        # cas limite : risques de l'IA générative, mais dans un chunk sur la qualité des données
        "BNS_annual_report_FY2024-0110": True},       # idem : « AI Risk Guidelines », risques de l'IA générative
    7: {"RBC_annual_report_FY2024-0176": True, "RBC_annual_report_FY2024-0102": True,
        "RBC_annual_report_FY2024-0177": False, "RBC_annual_report_FY2024-0142": False,
        "RBC_annual_report_FY2024-0124": False, "RBC_annual_report_FY2024-0178": False,
        "RBC_annual_report_FY2024-0213": False},
    8: {"TD_annual_report_FY2024-0035": False, "TD_annual_report_FY2024-0304": False,
        "TD_annual_report_FY2024-0055": False, "TD_annual_report_FY2024-0266": False,
        "TD_annual_report_FY2024-0239": False},
    9: {"BMO_annual_report_FY2024-0108": False, "BMO_annual_report_FY2024-0113": False,
        "BMO_annual_report_FY2024-0114": True,        # cas limite : hypothèques par région, assurées ou non, LTV moyen
        "BMO_annual_report_FY2024-0116": False},      # cas limite : tableau LTV des marges de crédit hypothécaires (HELOC)
    10: {"BNS_annual_report_FY2024-0005": False, "BNS_annual_report_FY2024-0105": False,
         "BNS_annual_report_FY2024-0007": False, "BNS_annual_report_FY2024-0280": False,
         "BNS_annual_report_FY2024-0156": False, "BNS_annual_report_FY2024-0107": True,
         "BNS_annual_report_FY2024-0108": True, "BNS_annual_report_FY2024-0103": False,
         "BNS_annual_report_FY2024-0122": False},
}

In [33]:
def verdict(qid, cid):
    return JUDGE.get(qid, {}).get(cid)           # None = pas encore jugé

def methods_in(res_dict):
    return [m for m in ("topk", "mmr") if any(k[1] == m for k in res_dict)]

def p3_table(res_dict):
    rows = []
    for Q in QUERIES:
        for m in methods_in(res_dict):
            v = [verdict(Q["id"], r["chunk_id"]) for r in res_dict[(Q["id"], m)][:3]]
            rows.append({"Q": Q["id"], "type": Q["type"], "méthode": m,
                         "pertinents": sum(x is True for x in v), "non jugés": sum(x is None for x in v)})
    d = pd.DataFrame(rows)
    d["P@3"] = d.pertinents / 3
    return d

def summarize(d, label):
    out = []
    for m in d["méthode"].unique():
        s = d[d["méthode"] == m]
        out.append({"version": label, "méthode": m, "P@3": round(s["P@3"].mean(), 2),
                    "pertinents /30": int(s.pertinents.sum()),
                    "questions descriptives": round(s[s.type == "broad"]["P@3"].mean(), 2),
                    "questions à chiffre": round(s[s.type == "factual"]["P@3"].mean(), 2),
                    "non jugés": int(s["non jugés"].sum())})
    return out

def review_unjudged(res_dict, label):
    seen = set()
    print(f"\n--- Chunks à juger ({label}) : ajouter un verdict dans JUDGE ---")
    for Q in QUERIES:
        for m in methods_in(res_dict):
            for r in res_dict[(Q["id"], m)][:3]:
                key = (Q["id"], r["chunk_id"])
                if verdict(*key) is None and key not in seen:
                    seen.add(key)
                    print(f"Q{Q['id']} | {r['chunk_id']} | …{snippet(r['text'], Q['kw'])}…")

versions = [("v1 (avec le nom de la banque)", results), ("v2 (sans le nom de la banque)", results_v2)]
tables, summary_rows = {}, []
for label, rd in versions:
    tables[label] = p3_table(rd)
    summary_rows += summarize(tables[label], label)
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

for label, rd in versions:
    if tables[label]["non jugés"].sum():
        review_unjudged(rd, label)
for label, d in tables.items():
    print(f"\nPrecision@3 par question — {label}")
    print(d.pivot(index="Q", columns="méthode", values="P@3").round(2).to_string())

                      version méthode  P@3  pertinents /30  questions descriptives  questions à chiffre  non jugés
v1 (avec le nom de la banque)    topk 0.33              10                    0.47                 0.20          0
v1 (avec le nom de la banque)     mmr 0.23               7                    0.33                 0.13          0
v2 (sans le nom de la banque)    topk 0.43              13                    0.67                 0.20          0
v2 (sans le nom de la banque)     mmr 0.40              12                    0.67                 0.13          0

Precision@3 par question — v1 (avec le nom de la banque)
méthode   mmr  topk
Q                  
1        0.00  0.00
2        0.33  0.33
3        0.67  1.00
4        0.00  0.00
5        0.00  0.00
6        0.67  1.00
7        0.33  0.67
8        0.00  0.00
9        0.33  0.33
10       0.00  0.00

Precision@3 par question — v2 (sans le nom de la banque)
méthode   mmr  topk
Q                  
1        0.00  0.00
2        

## 7. Diagnostic : pourquoi certaines questions échouent

Pour les questions les plus faibles, on repère dans le MD&A de la bonne banque et de la bonne année les chunks qui **contiennent le sujet** (le mot-clé revient plusieurs fois : c'est un indicateur automatique, pas un jugement de pertinence), puis on mesure leur **rang** dans la recherche, avec et sans le nom de la banque dans la question. On vérifie aussi si le mot-clé se trouve **au-delà du token 512**, donc dans la partie que MiniLM ne lit pas.

In [34]:
GOLD_KW = {2: r"liquidity coverage ratio|\bLCR\b", 4: r"cyber", 5: r"\bCET1\b", 8: r"economic value of equity|\bEVE\b", 10: r"emerging risk"}
MIN_HITS = {2: 3, 4: 3, 5: 3, 8: 2, 10: 1}

def subset_ranks(idx, q):
    s = emb[idx] @ embed_query(q)
    order = (-s).argsort()
    return {idx[i]: r + 1 for r, i in enumerate(order)}

rows = []
for Q in QUERIES:
    qid = Q["id"]
    if qid not in GOLD_KW:
        continue
    sub = chunks[(chunks.bank_code == Q["bank"]) & (chunks.fiscal_year == Q["year"])]
    idx = sub.index.to_numpy()
    kre = re.compile(GOLD_KW[qid], re.I)
    hits = sub.text.map(lambda t: len(kre.findall(t)))
    gold = sub[(hits >= MIN_HITS[qid]) & (sub.part == "MD&A")].assign(hits=hits).nlargest(3, "hits")
    r1, r2 = subset_ranks(idx, Q["q"]), subset_ranks(idx, QUERIES_V2[qid])
    for i, g in gold.iterrows():
        m = kre.search(g.text)
        pos = len(model.tokenizer(g.text[:m.start()], add_special_tokens=False)["input_ids"])
        rows.append({"Q": qid, "chunk_id": g.chunk_id, "occurrences": int(g.hits), "position du mot-clé (tokens)": pos,
                     "rang avec le nom de la banque": r1[i], "rang sans le nom": r2[i], "sur": len(sub)})
diag = pd.DataFrame(rows)
print(diag.to_string(index=False))
print(f"\nRang amélioré en retirant le nom de la banque : {(diag['rang sans le nom'] < diag['rang avec le nom de la banque']).sum()} / {len(diag)}")
print(f"Mot-clé situé après le token {MAX_SEQ} (non lu par MiniLM) : {(diag['position du mot-clé (tokens)'] > MAX_SEQ).sum()} / {len(diag)}")

 Q                       chunk_id  occurrences  position du mot-clé (tokens)  rang avec le nom de la banque  rang sans le nom  sur
 2   TD_annual_report_FY2025-0181           10                            30                              5                 3  401
 2   TD_annual_report_FY2025-0182            8                             2                              2                 1  401
 4  BNS_annual_report_FY2025-0108           12                           414                             17                 1  358
 4  BNS_annual_report_FY2025-0159            6                           406                             32                10  358
 5 CIBC_annual_report_FY2024-0074            8                            93                            258               153  354
 5 CIBC_annual_report_FY2024-0075            6                             0                              2                 1  354
 5 CIBC_annual_report_FY2024-0073            5                           227       

## 8. Filtrage par métadonnées (consigne 7)

In [35]:
q = "provision for credit losses and impaired loans"      # sans nom de banque, comme en v2
cases = {
    "RBC seulement (filtre banque + catégorie)": {"$and": [{"bank_code": "RBC"}, {"risk_category": "Credit risk"}]},
    "Toutes les banques (filtre catégorie seulement)": {"risk_category": "Credit risk"},
    "Aucun filtre": None,
}
for label, where in cases.items():
    res = retrieve_topk(q, k=5, where=where)
    print(f"\n=== {label} ===")
    for r in res:
        print(f"  sim {r['sim']:.2f} | {r['bank_code']:4} {r['fiscal_year']} | {r['risk_category']:<14} | {r['citation'].split(' – ')[2][:45]}")
    print("  Banques :", sorted({r['bank_code'] for r in res}))


=== RBC seulement (filtre banque + catégorie) ===
  sim 0.62 | RBC  2024 | Credit risk    | Critical Accounting Policies And Estimates (p
  sim 0.61 | RBC  2024 | Credit risk    | Note 2 Summary of material accounting policie
  sim 0.58 | RBC  2025 | Credit risk    | Critical Accounting Policies And Estimates (p
  sim 0.57 | RBC  2024 | Credit risk    | Note 5 Loans and allowance for credit losses 
  sim 0.57 | RBC  2025 | Credit risk    | Note 2 Summary of material accounting policie
  Banques : ['RBC']

=== Toutes les banques (filtre catégorie seulement) ===
  sim 0.71 | CIBC 2025 | Credit risk    | Credit Risk (p.75-76)
  sim 0.71 | CIBC 2024 | Credit risk    | Credit Risk (p.81-82)
  sim 0.70 | BMO  2025 | Credit risk    | MD&A (p.33-34)
  sim 0.69 | BNS  2024 | Credit risk    | Management’s Discussion And Analysis Highligh
  sim 0.68 | BNS  2025 | Credit risk    | Risk Management (p.95-96)
  Banques : ['BMO', 'BNS', 'CIBC']

=== Aucun filtre ===
  sim 0.71 | CIBC 2025 | Credit ri

## 9. Comparaison avec les embeddings OpenAI (consigne 8, optionnelle)

La consigne demande de comparer si on utilise aussi `text-embedding-3-small`. **Cette section ne s'exécute que si une clé `OPENAI_API_KEY` est présente dans `module4_genai_bank_rag/.env`** ; sinon elle s'arrête proprement et le reste du notebook est inchangé. Coût estimé pour tout le corpus : environ 0,06 $. À la différence de MiniLM, ce modèle accepte jusqu'à 8 191 tokens : il ne tronque pas nos chunks.

Les chunks que MiniLM n'avait pas retournés n'ont pas de verdict : ils sont listés, à juger dans `JUDGE_OPENAI`. La précision n'est comparable qu'une fois tous les chunks jugés.

In [36]:
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
RUN_OPENAI = os.getenv("OPENAI_API_KEY", "").startswith("sk-")
OAI_MODEL = "text-embedding-3-small"
JUDGE_OPENAI = {}        # verdicts supplémentaires, même format que JUDGE : {question: {chunk_id: True/False}}

if not RUN_OPENAI:
    print("Aucune clé OpenAI détectée : section sautée (la consigne 8 est facultative).")
else:
    import tiktoken
    from openai import OpenAI
    oai = OpenAI()
    enc = tiktoken.get_encoding("cl100k_base")
    oai_path = MODELS / "openai_te3small_embeddings.npy"

    if oai_path.exists() and not FORCE_RECOMPUTE and np.load(oai_path).shape[0] == len(chunks):
        emb_oai = np.load(oai_path)
        print("Embeddings OpenAI rechargés depuis models/")
    else:
        texts = chunks.text.tolist()
        n_tok, parts, t0 = 0, [], time.time()
        for i in range(0, len(texts), 96):
            batch = texts[i:i + 96]
            n_tok += sum(len(enc.encode(t, disallowed_special=())) for t in batch)
            parts += [d.embedding for d in oai.embeddings.create(model=OAI_MODEL, input=batch).data]
        emb_oai = np.array(parts, dtype="float32")
        np.save(oai_path, emb_oai)
        print(f"{len(parts)} embeddings en {time.time() - t0:.0f} s | {n_tok:,} tokens | coût : ${n_tok / 1e6 * 0.02:.3f}")
    emb_oai = emb_oai / np.linalg.norm(emb_oai, axis=1, keepdims=True)

    COL_OAI = "bank_annual_reports_openai"
    if COL_OAI in [c.name for c in client.list_collections()]:
        client.delete_collection(COL_OAI)
    col_oai = client.create_collection(COL_OAI, metadata={"hnsw:space": "cosine"})
    for i in range(0, len(chunks), 500):
        sl = slice(i, i + 500)
        col_oai.add(ids=chunks.chunk_id.tolist()[sl], embeddings=emb_oai[sl].tolist(),
                    documents=chunks.text.tolist()[sl], metadatas=metadatas[sl])

    results_oai = {}
    for Q in QUERIES:
        qv = np.array(oai.embeddings.create(model=OAI_MODEL, input=[QUERIES_V2.get(Q["id"], Q["q"])]).data[0].embedding)
        qv = qv / np.linalg.norm(qv)
        res = col_oai.query(query_embeddings=[qv.tolist()], n_results=K, where=build_where(Q["bank"], Q["year"]),
                            include=["documents", "metadatas", "distances"])
        results_oai[(Q["id"], "topk")] = to_rows(res)

    for q_, d_ in JUDGE_OPENAI.items():
        JUDGE.setdefault(q_, {}).update(d_)
    d_oai = p3_table(results_oai)
    print(pd.DataFrame(summarize(d_oai, "OpenAI text-embedding-3-small (requêtes v2)")).to_string(index=False))
    if d_oai["non jugés"].sum():
        review_unjudged(results_oai, "OpenAI")
    print("\nPour comparaison, MiniLM v2 :")
    print(summary_df[summary_df.version.str.startswith("v2") & (summary_df["méthode"] == "topk")].to_string(index=False))

Aucune clé OpenAI détectée : section sautée (la consigne 8 est facultative).


## 10. Observations et conclusions

*(chiffres du run du 15 octobre 2026)*

### Embeddings et base vectorielle
- **Modèle :** `all-MiniLM-L6-v2`, local, gratuit, 384 dimensions, vecteurs normalisés. 3 562 chunks embeddés en 6,8 minutes sur CPU, coût 0 $.
- **Troncature :** nos chunks font 724 tokens en moyenne avec le tokenizer de MiniLM. **90 % dépassent 512 tokens** et le modèle ne voit que **69 % du texte**. Le texte complet reste stocké et sera donné au LLM, mais le vecteur de recherche ne représente que le début de chaque chunk. Le diagnostic de la section 7 n'attribue pourtant pas l'essentiel des échecs à cette cause (1 chunk cible sur 12 a son mot-clé après le token 512).
- **ChromaDB :** collection persistante, similarité cosinus, 3 562 chunks. Les comptes par banque et par année correspondent à ceux du notebook 01.
- **Sans filtre**, une recherche peut renvoyer la mauvaise banque : le plus proche voisin d'un état financier de BNS était celui de CIBC (similarité 0,877).

### Top-k contre MMR
- MMR diversifie peu : 2 à 5 chunks sur 5 en commun avec top-k. Identique pour Q9, et il remplace deux bons résultats par des résultats moins proches pour Q7.
- Sur la question sans filtre (Q6), MMR renvoie 3 banques contre 2 pour top-k.
- Le chevauchement entre chunks fait parfois remonter deux fois le même passage (Q9).
- **Décision :** top-k par défaut ; MMR seulement quand la diversité compte.

### Seuil de similarité (0,3)
- Il écarte les questions hors domaine (« capitale de la France » : 0,185).
- Il **ne détecte pas une banque absente de la base** : « CET1 de Desjardins » atteint 0,408 et renvoie des chunks de BMO. À traiter au notebook 04 (vérifier le nom de la banque avant la recherche).

### Precision@3 (10 requêtes, jugement manuel)

| | v1 (avec le nom de la banque) | v2 (sans le nom) |
|---|---|---|
| Top-k | 0,33 (10/30) | 0,43 (13/30) |
| MMR | 0,23 (7/30) | 0,40 (12/30) |

- **Questions descriptives** (Q3, Q4, Q6, Q9, Q10) : top-k 0,47 → 0,67 ; MMR 0,33 → 0,67. Le gain vient surtout de Q4 et Q10.
- **Questions à chiffre** (Q1, Q2, Q5, Q7, Q8) : stables, autour de 0,20 (top-k) et 0,13 (MMR). Q1, Q5 et Q8 sont à 0 dans les deux versions.
- **Cause du gain en v2 :** les chunks écrivent « the Bank » plutôt que « Scotiabank » ou « CIBC », et le nom attire des pages d'information aux actionnaires. Le filtre de métadonnées fixe déjà la banque.
- **Cause probable des questions à chiffre faibles (non testée) :** les tableaux sont extraits à plat (« $ 215 $ 126 $ 231 »), sans alignement de colonnes ; un chiffre peut aussi se retrouver dans un chunk voisin de son explication (LCR de TD, CET1 de CIBC).

### Filtrage par métadonnées
Requête « provision for credit losses and impaired loans », top-5 : avec le filtre banque + catégorie, 5 résultats RBC ; avec la catégorie seule, 5 résultats de BMO, BNS et CIBC, **aucun de RBC ni de TD**. Sans filtre de banque, une question sur RBC ne renvoie donc aucun passage de RBC : le filtre est indispensable. Les similarités plus élevées sans filtre (0,68 à 0,71 contre 0,57 à 0,62) ne mesurent pas la qualité.

### Limites
- **30 jugements seulement** : un écart de 3 à 5 passages ne prouve rien de solide.
- **La v2 a été écrite après avoir vu les échecs de la v1** : l'amélioration est probablement surestimée.
- **Les verdicts sont à relire** : ils s'appuient sur des extraits et des recherches de mots-clés ; cas limites : RBC FY2025-0151, BMO FY2024-0114 et 0116, BNS FY2025-0112 et FY2024-0110.
- **Les titres de section de RBC et BMO sont approximatifs** (pas de signets dans leurs PDF), et la catégorie de risque est une heuristique à mots-clés.
- Une seule requête testée pour le filtrage ; la pertinence des résultats RBC n'a pas été jugée.

### Décisions pour la suite
1. Notebook 04 : extraire la banque et l'année de la question pour construire le filtre, et chercher sur le reste de la question (sans le nom de la banque).
2. Pistes pour les questions à chiffre : recherche hybride (mots-clés + embeddings), re-ranking, extraction des tableaux.
3. Notebook 03 : réutiliser `models/chroma/` (collection `bank_annual_reports_minilm`) et les fonctions de la section 4 (à déplacer dans `src/`).

In [37]:
# Sauvegarde des résultats de ce notebook
summary_df.to_csv(REPORTS / "precision_at_3.csv", index=False)
pd.DataFrame(RETRIEVAL_LOG).to_json(REPORTS / "retrieval_log.jsonl", orient="records", lines=True, force_ascii=False)
print(f"reports/precision_at_3.csv | reports/retrieval_log.jsonl ({len(RETRIEVAL_LOG)} lignes) | models/ : embeddings et collection ChromaDB")

reports/precision_at_3.csv | reports/retrieval_log.jsonl (200 lignes) | models/ : embeddings et collection ChromaDB
